# ClearCare: Python + DuckDB + SQL walkthrough

This notebook explains the reproducible workflow. Install `requirements.txt` in your chosen notebook environment first. It uses public project data, not personal medical records. Run from this folder or the repository root.


In [ ]:
from pathlib import Path
import sys
import duckdb

project_root = Path.cwd()
if not (project_root / "sql").exists():
    project_root = project_root.parent
if not (project_root / "sql/schema.sql").exists():
    raise RuntimeError("Run from the repository root or notebooks/ folder.")
sys.path.insert(0, str(project_root))
from clearcare.core import build_database, rows, sql, Context, generate_guide, render_markdown

build_database(project_root / "build/notebook_demo.duckdb")
con = duckdb.connect(str(project_root / "build/notebook_demo.duckdb"), read_only=True)


## 1. Inspect a schema

Unlike a CSV alone, a table has explicit types and keys. The source ID connects facts to an official page. The builder already checked all reviewed input hashes.


In [ ]:
rows(con, "DESCRIBE sources")


## 2. Join four tables

The result shows advertised imaging information and its scope. It does not confirm that a particular test is available or covered. Named SQL parameters separate values from query text.


In [ ]:
imaging = rows(con, sql("facility_services.sql"), {"service_id": "radiology", "as_of": "2026-10-07"})
imaging


## 3. Aggregate source coverage

These counts describe the curated sample, not the healthcare system as a whole.


In [ ]:
rows(con, "SELECT publisher, count(*) AS source_count FROM sources GROUP BY publisher ORDER BY source_count DESC, publisher")


## 4. Inspect missing network evidence

A LEFT JOIN keeps locations that lack observations. Zero observations means unknown, not out of network.


In [ ]:
rows(con, """
SELECT f.name, count(o.observation_id) AS network_observations
FROM facilities f
LEFT JOIN network_observations o USING (facility_id)
GROUP BY f.name
ORDER BY f.name
""")


## 5. Generate a conditional guide

Python controls the scenario and preparation flags; SQL chooses the steps. The guide asks the original office about records when records are unavailable.


In [ ]:
guide = generate_guide(con, Context(needs_interpreter=True, has_prior_records=False), "2026-10-07")
assert guide["coverage_confirmed"] is False
assert "records_missing" in [step["step_id"] for step in guide["steps"]]
print(render_markdown(guide))


## 6. Recheck the same snapshot at a later date

The source summaries do not become current merely because the software is rerun. Old instructions are replaced with a review prompt.


In [ ]:
later = generate_guide(con, Context(), "2027-11-01")
assert all(step["evidence_state"] == "needs_review" for step in later["steps"])
[(step["step_id"], step["evidence_state"]) for step in later["steps"]]


## 7. Close the connection

This allows another process to rebuild the derived database without file-lock conflicts. SQL execution and saving SQL source are different actions; the published queries remain in `sql/`.


In [ ]:
con.close()
